In [28]:
import re
import nltk
from nltk.stem import PorterStemmer
from nltk.corpus import stopwords
import requests
from bs4 import BeautifulSoup

# Download necessary NLTK data
nltk.download('punkt', quiet=True)
nltk.download('stopwords', quiet=True)

url = "https://en.wikipedia.org/wiki/Plant_disease"
global_stemmer = PorterStemmer()
stop_words = set(stopwords.words('english'))

def fetch_page(url):
    headers = {"User-Agent": "Mozilla/5.0"}
    try:
        response = requests.get(url, headers=headers)
        if response.status_code == 200:
            return BeautifulSoup(response.text, "html.parser")
        return None
    except Exception as e:
        print("Request error:", e)
        return None

def index_words_with_context(soup, target_word="disease"):
    index = {}
    context_matches = []

    # Remove non-visible elements
    for script_or_style in soup(["script", "style"]):
        script_or_style.decompose()

    # Get text with space separator to avoid word merging
    text = soup.get_text(separator=' ', strip=True)

    # Find all words using a simple alphabetical pattern
    words = re.findall(r'[a-z]+', text.lower())

    for word in words:
        index[word] = index.get(word, 0) + 1

    # For the audit: find context using regex to show surrounding text
    # This helps verify the browser's 'find' results
    pattern = re.compile(r'(.{0,30}\b' + re.escape(target_word) + r's?\b.{0,30})', re.IGNORECASE)
    context_matches = pattern.findall(text)

    return index, context_matches

def apply_stemming_and_filter(initial_index):
    stemmed_index = {}
    debug_map = {}
    for word, count in initial_index.items():
        if word in stop_words or len(word) < 2:
            continue
        stem = global_stemmer.stem(word)
        stemmed_index[stem] = stemmed_index.get(stem, 0) + count
        if stem not in debug_map:
            debug_map[stem] = {}
        debug_map[stem][word] = count
    return stemmed_index, debug_map

def search(query, index):
    query = query.lower().strip()
    if ' and ' in query:
        parts = [global_stemmer.stem(p.strip()) for p in query.split(' and ')]
        matches = {p: index.get(p, 0) for p in parts if p in index}
        score = sum(matches.values()) if len(matches) == len(parts) else 0
        return {'matched_words': matches if score > 0 else {}, 'rank_score': score}
    elif ' or ' in query:
        parts = [global_stemmer.stem(p.strip()) for p in query.split(' or ')]
        matches = {p: index.get(p, 0) for p in parts if p in index}
        return {'matched_words': matches, 'rank_score': sum(matches.values())}
    else:
        stem = global_stemmer.stem(query)
        count = index.get(stem, 0)
        return {'matched_words': {stem: count} if count > 0 else {}, 'rank_score': count}

soup = fetch_page(url)
if soup:
    raw_index, context_list = index_words_with_context(soup, "disease")
    final_index, debug_info = apply_stemming_and_filter(raw_index)

    target_stem = global_stemmer.stem('disease')
    print(f"--- Detailed Audit for Stem: '{target_stem}' ---")
    if target_stem in debug_info:
        for word, count in debug_info[target_stem].items():
            print(f"'{word}': {count}")
        print(f"Total: {final_index[target_stem]}")

        print(f"\n--- Contextual proof (First 10 occurrences) ---")
        for i, ctx in enumerate(context_list[:10]):
            print(f"{i+1}: ...{ctx}...")

    Queries = ["Disease", "Detection AND Plant"]
    for q in Queries:
        print(f"\nQuery: '{q}' -> {search(q, final_index)}")

--- Detailed Audit for Stem: 'diseas' ---
'disease': 25
'diseases': 19
Total: 44

--- Contextual proof (First 10 occurrences) ---
1: ...Plant disease - Wikipedia Jump to content M...
2: ...e the table of contents Plant disease 19 languages العربية Българск...
3: ...ipedia, the free encyclopedia Diseases of plants For the journal, se...
4: ...e Plant Disease (journal) . Life cycle of the...
5: ...ris pathovar campestris Plant diseases are diseases in plants caused...
6: ...ganisms that cause infectious disease include fungi , oomycetes , b...
7: ...pathogens. The study of plant disease is called plant pathology . P...
8: ...r information: Lists of plant diseases Fungi [ edit ] Powdery mildew...
9: ...cultative saprotrophs. Fungal diseases may be controlled through the...
10: ... Fusarium spp. (Fusarium wilt disease) Magnaporthe grisea (rice bla...

Query: 'Disease' -> {'matched_words': {'diseas': 44}, 'rank_score': 44}

Query: 'Detection AND Plant' -> {'matched_words': {'detect': 3, 'pl